# Data preparation for the feature combination experiment

This notebook creates two outputs from the **experiment-only English corpus**:

- `games_cleaned.csv`: one row per retained game with its display/experiment metadata and one final `Description` column.
- `games_descriptors.csv`: a weighted binary bag-of-words matrix, keyed by `AppID`.

The experiment was explicitly scoped to `Translation Done/no_translation_needed.csv`. That is the default input below. The combined `games_description_english_only.csv` includes the translated batches and is for the later full-corpus workflow; change `INPUT_CSV` only when you intentionally want that broader corpus.

The descriptor vocabulary is the union of Tags, Genres, and Categories. Tags retain their existing order and use the fitted length-aware tag weights from `Tag Weight Study/tag_weights.py`. Genres and Categories have value 1 when present. If the same exact descriptor appears in more than one source, the cell takes the maximum value; thus a Genre/Category occurrence keeps it at 1. Descriptors absent from a game are 0.

The descriptor export is a wide CSV matrix and therefore writes explicit zeros. Its size can be substantial. This notebook streams input rows so it does not need to hold the full dense matrix in memory.


In [ ]:
from pathlib import Path
import sys
import csv
import math
import pandas as pd

# Locate the project whether Jupyter starts in the project root or Data Prep.
_search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in _search_roots
     if (path / 'Data Prep').is_dir() and (path / 'Tag Weight Study').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the project root (expected Data Prep/ and Tag Weight Study/).'
    )

DATA_PREP_DIR = PROJECT_ROOT / 'Data Prep'
TAG_WEIGHT_DIR = PROJECT_ROOT / 'Tag Weight Study'

# Experiment-only corpus, as agreed. To prepare the later translated corpus,
# intentionally change this to DATA_PREP_DIR / 'games_description_english_only.csv'.
INPUT_CSV = DATA_PREP_DIR / 'Translation Done' / 'no_translation_needed.csv'
RAW_GAMES_CSV = DATA_PREP_DIR / 'games.csv'  # restores Peak CCU dropped earlier
CLEANED_OUTPUT_CSV = DATA_PREP_DIR / 'games_cleaned.csv'
DESCRIPTORS_OUTPUT_CSV = DATA_PREP_DIR / 'games_descriptors.csv'

if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_CSV}')
if not RAW_GAMES_CSV.exists():
    raise FileNotFoundError(f'Raw catalogue needed to restore Peak CCU was not found: {RAW_GAMES_CSV}')

sys.path.insert(0, str(TAG_WEIGHT_DIR))
from tag_weights import weight_tags

print(f'Project root: {PROJECT_ROOT}')
print(f'Experiment input: {INPUT_CSV}')
print(f'Cleaned output: {CLEANED_OUTPUT_CSV}')
print(f'Descriptor output: {DESCRIPTORS_OUTPUT_CSV}')


In [ ]:
# Check the expected schema and retrieve Peak CCU from the original catalogue.
SOURCE_COLUMNS = [
    'AppID', 'Name', 'About the game', 'Tags', 'Genres', 'Categories',
    'Developers', 'Publishers', 'Price',
]

input_columns = set(pd.read_csv(INPUT_CSV, nrows=0, encoding='utf-8-sig').columns)
missing = [column for column in SOURCE_COLUMNS if column not in input_columns]
if missing:
    raise KeyError(f'Expected columns missing from experiment input: {missing}')

raw_columns = set(pd.read_csv(RAW_GAMES_CSV, nrows=0, encoding='utf-8-sig').columns)
if not {'AppID', 'Peak CCU'}.issubset(raw_columns):
    raise KeyError('Raw games.csv must contain AppID and Peak CCU columns.')

peak_lookup = pd.read_csv(
    RAW_GAMES_CSV,
    usecols=['AppID', 'Peak CCU'],
    dtype={'AppID': 'string'},
    encoding='utf-8-sig',
    low_memory=False,
)
peak_lookup['AppID'] = peak_lookup['AppID'].str.strip()
peak_lookup = peak_lookup.dropna(subset=['AppID'])

# A duplicate AppID is safe only if every raw row agrees on its Peak CCU.
conflicting_peak_ids = (
    peak_lookup.groupby('AppID', dropna=False)['Peak CCU']
    .nunique(dropna=False)
    .loc[lambda values: values > 1]
)
if not conflicting_peak_ids.empty:
    raise ValueError(
        f'{len(conflicting_peak_ids)} AppIDs have conflicting Peak CCU values in games.csv. '
        'Resolve these before making the cleaned export.'
    )
peak_lookup = peak_lookup.drop_duplicates('AppID').set_index('AppID')['Peak CCU']
print(f'Peak CCU lookup loaded for {len(peak_lookup):,} unique AppIDs.')

# Peak CCU is highly zero-inflated and has a long right tail. Use fixed
# order-of-magnitude bands so blockbuster games are not grouped into a broad
# low-CCU quantile bucket. Zero is kept as its own level.
POPULARITY_BIN_EDGES = [0, 10, 100, 1_000, 10_000, 100_000, math.inf]
POPULARITY_POSITIVE_LABELS = [
    'Very low (1-10)',
    'Low (11-100)',
    'Moderate (101-1,000)',
    'High (1,001-10,000)',
    'Very high (10,001-100,000)',
    'Extreme (100,001+)',
]


def popularity_bin(value):
    if pd.isna(value):
        return 'Unknown'
    value = float(value)
    if value < 0:
        raise ValueError(f'Negative Peak CCU cannot be binned: {value}')
    if value == 0:
        return 'Zero (0)'
    label = pd.cut(
        pd.Series([value]),
        bins=POPULARITY_BIN_EDGES,
        labels=POPULARITY_POSITIVE_LABELS,
        include_lowest=True,
        right=True,
    ).iloc[0]
    if pd.isna(label):
        raise ValueError(f'Peak CCU value fell outside the configured bins: {value}')
    return str(label)

print(f'Popularity categories: {len(POPULARITY_POSITIVE_LABELS) + 1} (including zero)')
print(f'Peak CCU bin edges: {POPULARITY_BIN_EDGES}')


In [ ]:
def split_descriptors(value):
    """Split comma-separated values, trim whitespace, and preserve order."""
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(',') if part.strip()]


def has_text(value):
    return not pd.isna(value) and bool(str(value).strip())

# First streaming pass: determine one consistent feature vocabulary and report rows
# that would be excluded by the agreed empty-description/empty-descriptor rules.
descriptor_vocabulary = set()
source_vocabularies = {'Tags': set(), 'Genres': set(), 'Categories': set()}
overlap_game_counts = {('Tags', 'Genres'): 0, ('Tags', 'Categories'): 0, ('Genres', 'Categories'): 0}
all_three_overlap_games = 0
source_rows = 0
rows_empty_description = 0
rows_empty_all_descriptor_sources = 0

for chunk in pd.read_csv(
    INPUT_CSV,
    usecols=SOURCE_COLUMNS,
    dtype={'AppID': 'string'},
    encoding='utf-8-sig',
    chunksize=20_000,
    low_memory=False,
):
    source_rows += len(chunk)
    for row in chunk.itertuples(index=False, name=None):
        record = dict(zip(SOURCE_COLUMNS, row))
        tags = split_descriptors(record['Tags'])
        genres = split_descriptors(record['Genres'])
        categories = split_descriptors(record['Categories'])
        tag_set, genre_set, category_set = set(tags), set(genres), set(categories)
        descriptor_vocabulary.update(tags)
        descriptor_vocabulary.update(genres)
        descriptor_vocabulary.update(categories)
        source_vocabularies['Tags'].update(tag_set)
        source_vocabularies['Genres'].update(genre_set)
        source_vocabularies['Categories'].update(category_set)
        overlap_game_counts[('Tags', 'Genres')] += bool(tag_set & genre_set)
        overlap_game_counts[('Tags', 'Categories')] += bool(tag_set & category_set)
        overlap_game_counts[('Genres', 'Categories')] += bool(genre_set & category_set)
        all_three_overlap_games += bool(tag_set & genre_set & category_set)
        if not has_text(record['About the game']):
            rows_empty_description += 1
        if not (tags or genres or categories):
            rows_empty_all_descriptor_sources += 1

# Sort columns for reproducible output; this does not change the rank order used
# to calculate tag weights within each individual game.
descriptor_terms = sorted(descriptor_vocabulary, key=lambda value: (value.casefold(), value))
print(f'Input rows: {source_rows:,}')
print(f'Unique descriptor terms: {len(descriptor_terms):,}')
print(f'Rows with empty final description: {rows_empty_description:,}')
print(f'Rows with Tags, Genres, and Categories all empty: {rows_empty_all_descriptor_sources:,}')
for source_a, source_b in overlap_game_counts:
    shared_terms = source_vocabularies[source_a] & source_vocabularies[source_b]
    print(f'{source_a} / {source_b}: {len(shared_terms):,} shared terms; {overlap_game_counts[(source_a, source_b)]:,} games overlap')
print(f'Games with an exact term in all three sources: {all_three_overlap_games:,}')


In [ ]:
# Second streaming pass: write aligned cleaned-game and descriptor files.
# Rows with empty descriptions or no descriptor source are dropped, following
# the preprocessing decisions already made for the project.
CLEANED_COLUMNS = [
    'AppID', 'Name', 'Description', 'Peak CCU', 'Popularity Bin', 'Developers', 'Publishers', 'Price',
]
DESCRIPTOR_COLUMNS = ['AppID', *descriptor_terms]

written_rows = 0
missing_peak_ccu_ids = 0
popularity_bin_counts = {}
seen_app_ids = set()

with CLEANED_OUTPUT_CSV.open('w', encoding='utf-8-sig', newline='') as cleaned_file, \
     DESCRIPTORS_OUTPUT_CSV.open('w', encoding='utf-8-sig', newline='') as descriptor_file:
    cleaned_writer = csv.writer(cleaned_file)
    descriptor_writer = csv.writer(descriptor_file)
    cleaned_writer.writerow(CLEANED_COLUMNS)
    descriptor_writer.writerow(DESCRIPTOR_COLUMNS)

    for chunk in pd.read_csv(
        INPUT_CSV,
        usecols=SOURCE_COLUMNS,
        dtype={'AppID': 'string'},
        encoding='utf-8-sig',
        chunksize=20_000,
        low_memory=False,
    ):
        for row in chunk.itertuples(index=False, name=None):
            record = dict(zip(SOURCE_COLUMNS, row))
            app_id = '' if pd.isna(record['AppID']) else str(record['AppID']).strip()
            if not app_id:
                continue
            if app_id in seen_app_ids:
                raise ValueError(f'Duplicate AppID in experiment input: {app_id}')
            seen_app_ids.add(app_id)

            description = record['About the game']
            tags = split_descriptors(record['Tags'])
            genres = split_descriptors(record['Genres'])
            categories = split_descriptors(record['Categories'])
            if not has_text(description) or not (tags or genres or categories):
                continue

            # Tag rank is the original left-to-right order in the catalogue.
            # Genres and Categories have fixed value 1.0. If an exact descriptor
            # appears in multiple sources, max() preserves the strongest value.
            weighted_values = {}
            for term, weight in weight_tags(tags):
                weighted_values[term] = max(weighted_values.get(term, 0.0), float(weight))
            for term in genres + categories:
                weighted_values[term] = max(weighted_values.get(term, 0.0), 1.0)

            peak_ccu = peak_lookup.get(app_id, pd.NA)
            if pd.isna(peak_ccu):
                missing_peak_ccu_ids += 1

            peak_ccu_category = popularity_bin(peak_ccu)
            popularity_bin_counts[peak_ccu_category] = popularity_bin_counts.get(peak_ccu_category, 0) + 1

            cleaned_writer.writerow([
                app_id,
                '' if pd.isna(record['Name']) else record['Name'],
                str(description).strip(),
                '' if pd.isna(peak_ccu) else peak_ccu,
                peak_ccu_category,
                '' if pd.isna(record['Developers']) else record['Developers'],
                '' if pd.isna(record['Publishers']) else record['Publishers'],
                '' if pd.isna(record['Price']) else record['Price'],
            ])

            descriptor_writer.writerow([
                app_id,
                *[format(weighted_values.get(term, 0.0), '.12g') for term in descriptor_terms],
            ])
            written_rows += 1

print(f'Rows written to each output: {written_rows:,}')
print(f'Input AppIDs without a matching Peak CCU value: {missing_peak_ccu_ids:,}')
print('Popularity bin counts:', popularity_bin_counts)
print(f'Cleaned games: {CLEANED_OUTPUT_CSV}')
print(f'Weighted descriptors: {DESCRIPTORS_OUTPUT_CSV}')


## Output schema and interpretation

`games_cleaned.csv` contains `AppID`, `Name`, `Description` (from the code-cleaned and Notes-appended `About the game` field), `Peak CCU` (joined back from raw `games.csv` by AppID), `Popularity Bin`, `Developers`, `Publishers`, and `Price`. Translation/audit bookkeeping fields and the separate original-description field are omitted.

`games_descriptors.csv` contains `AppID` plus one column per unique descriptor, named with the descriptor itself. A present tag has its rank-decay weight; a present genre or category has value 1; an absent descriptor has value 0. When the same exact term appears in multiple sources, the cell stores the maximum source value (so Genre/Category presence makes it 1).

Both files contain the same retained AppIDs in the same order. Use `Popularity Bin` as the categorical popularity feature in the feature-combination experiment; retain raw `Peak CCU` for display or later analysis.

Popularity binning uses fixed order-of-magnitude ranges: zero, 1-10, 11-100, 101-1,000, 1,001-10,000, 10,001-100,000, and 100,001+. This keeps the highly skewed upper tail visible; missing values are labeled `Unknown`.
